In [22]:
%%writefile train.py
import os
import torch
import torch.distributed as dist

def ddp_init():
    """Initialize DDP environment."""
    if "RANK" not in os.environ:
        raise RuntimeError("Run this script with torchrun")

    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    # Set device
    torch.cuda.set_device(local_rank)

    # Init process group
    dist.init_process_group(
        backend="nccl",
        init_method="env://",
        world_size=world_size,
        rank=rank,
    )

    return rank, local_rank, world_size


def main():
    rank, local_rank, world_size = ddp_init()

    # Print from each GPU
    print(f"[Rank {rank}] Hello from GPU {local_rank} / World size {world_size}")

    # Sync all processes
    dist.barrier()

    # Cleanup
    dist.destroy_process_group()


if __name__ == "__main__":
    main()


Overwriting train.py


In [23]:
!torchrun --nproc_per_node=2 train.py


W0101 14:33:47.599000 225 torch/distributed/run.py:774] 
W0101 14:33:47.599000 225 torch/distributed/run.py:774] *****************************************
W0101 14:33:47.599000 225 torch/distributed/run.py:774] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0101 14:33:47.599000 225 torch/distributed/run.py:774] *****************************************
[Rank 0] Hello from GPU 0 / World size 2
/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:4807: UserWarning: No device id is provided via `init_process_group` or `barrier `. Using the current device set by the user. 
  warnings.warn(  # warn only once
[rank0]:[W101 14:33:49.180479180 ProcessGroupNCCL.cpp:5023] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 as device used by this process is currently unknown. This can potentially cause a hang if this rank 

In [26]:
%%writefile train.py
import os
import torch
import torch.distributed as dist
from torch.distributed import init_process_group, destroy_process_group

def ddp_init():
    """Initialize DDP and return rank info."""
    # torchrun sets these automatically
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    assert torch.cuda.is_available(), "CUDA is required for this DDP test"

    # Set device for this process
    torch.cuda.set_device(local_rank)

    # Initialize process group
    init_process_group(
        backend="nccl",
        init_method="env://",
        rank=rank,
        world_size=world_size,
    )

    master_process = rank == 0
    return rank, local_rank, world_size, master_process


def cleanup():
    dist.destroy_process_group()


def main():
    rank, local_rank, world_size, master_process = ddp_init()

    if master_process:
        print("I'm main (rank 0)")

    print(f"Hello from rank {rank} on GPU {local_rank}")

    cleanup()


if __name__ == "__main__":
    main()


Overwriting train.py


In [27]:
!torchrun --nproc_per_node=2 train.py


W0101 14:36:06.437000 276 torch/distributed/run.py:774] 
W0101 14:36:06.437000 276 torch/distributed/run.py:774] *****************************************
W0101 14:36:06.437000 276 torch/distributed/run.py:774] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0101 14:36:06.437000 276 torch/distributed/run.py:774] *****************************************
Hello from rank 1 on GPU 1
I'm main (rank 0)
Hello from rank 0 on GPU 0
